In [296]:
from datasets import load_dataset
from src.language_models.model import RNNModel as lstm
import torch
from src.language_models.dictionary_corpus import Dictionary, Corpus, tokenize
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import numpy as np
import string

# Load dataset

In [3]:

ds = load_dataset("nyu-mll/blimp", "adjunct_island")

Generating train split: 100%|██████████| 1000/1000 [00:00<00:00, 231755.11 examples/s]


In [17]:
ds['train'][0]

{'sentence_good': 'Who should Derek hug after shocking Richard?',
 'sentence_bad': 'Who should Derek hug Richard after shocking?',
 'field': 'syntax',
 'linguistics_term': 'island_effects',
 'UID': 'adjunct_island',
 'simple_LM_method': True,
 'one_prefix_method': False,
 'two_prefix_method': False,
 'lexically_identical': True,
 'pair_id': 0}

# Load model

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/full_check/epoch_40.pt"  # Replace with your checkpoint path
model = lstm('LSTM', 50001, 200, 650, 2, 0.2, False)
with open(checkpoint_path, 'rb') as f:
    state_dict = torch.load(f, map_location='cuda' if device =='cuda' else 'cpu')
    model.load_state_dict(state_dict)
model.to(device)
model.eval() 

RNNModel(
  (drop): Dropout(p=0.2, inplace=False)
  (encoder): Embedding(50001, 200)
  (rnn): LSTM(200, 650, num_layers=2, dropout=0.2)
  (decoder): Linear(in_features=650, out_features=50001, bias=True)
)

# Load vocabulary and tokenize dataset

In [293]:
data_path = "/scratch2/mrenaudin/colorlessgreenRNNs/english_data"  # current directory
dictionary = Dictionary(data_path)

In [297]:
class BLiMPDataset(Dataset):
    def __init__(self, blimp_subset, dictionary):
        self.dataset = load_dataset("nyu-mll/blimp", blimp_subset, split = 'train')
        self.dictionary = dictionary
        self.encoded_pairs = []

        for example in self.dataset:
            sentence_good = example['sentence_good']
            sentence_bad = example['sentence_bad']
            sentence_good = sentence_good.rstrip(string.punctuation)
            sentence_bad = sentence_bad.rstrip(string.punctuation)
            encoded_good = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence_good.split()]
            #encoded_good = [self.dictionary.word2idx.get(word) for word in sentence_good.split()]
            encoded_bad = [self.dictionary.word2idx.get(word, self.dictionary.word2idx.get("<unk>")) for word in sentence_bad.split()]
            #encoded_bad = [self.dictionary.word2idx.get(word) for word in sentence_bad.split()]
            self.encoded_pairs.append({
                "sentence_good": sentence_good,
                "sentence_bad": sentence_bad,
                "encoded_good": torch.tensor(encoded_good, dtype=torch.long),
                "encoded_bad": torch.tensor(encoded_bad, dtype=torch.long),
            })

    def __len__(self):
        return len(self.encoded_pairs)

    def __getitem__(self, idx):
        return self.encoded_pairs[idx]

In [395]:
blimp = BLiMPDataset("irregular_past_participle_verbs", dictionary)

In [396]:
def collate_fn(batch):
    encoded_good_sequences = [item['encoded_good'] for item in batch]
    encoded_bad_sequences = [item['encoded_bad'] for item in batch]
    sentence_good = [item['sentence_good'] for item in batch]
    sentence_bad = [item['sentence_bad'] for item in batch]
    return {
        'sentence_bad':sentence_bad,
        'sentence_good': sentence_good,
        'encoded_good': pad_sequence(encoded_good_sequences, batch_first=True),
        'encoded_bad': pad_sequence(encoded_bad_sequences, batch_first=True)
    }


In [397]:
test_dataloader = DataLoader(blimp, batch_size=512, collate_fn = collate_fn)

In [398]:
def compute_seq_nll(data, hidden, batch_size, max_len):
    print('data', data)
    #mask
    mask = (data!=0).float()
    #forward pass
    
    pad = data.swapaxes(0,1)
    print('pad', pad.shape)
    
    output, hidden = model(pad, hidden)
    print('output', output.shape)
    #target
    targets = data[:, 1:]#.swapaxes(0,1)
    print('targets', targets.shape)
    #log probs
    log_probs = F.log_softmax(output, dim=-1)
    log_probs = log_probs[:-1]
    log_probs = log_probs.swapaxes(0,1)
    #nll loss
    #WITH NLL LOSS
    nll_loss = F.nll_loss(
            log_probs.reshape(-1, log_probs.size(-1)),
            targets.reshape(-1),
            reduction='none'
        )#.reshape(batch_size, max_len - 1)
    print('nll1', nll_loss.shape)
    nll_loss=nll_loss.reshape(batch_size, max_len - 1)
    print('nll2', nll_loss.shape)
    #mask loss
    masked_nll_loss = nll_loss * mask[:, 1:]
    # Sum the negative log-likelihood over the sequence for each example
    sequence_nll = masked_nll_loss.sum(dim=1)
    return -sequence_nll
    # vocab_size = output.shape[2]
    # log_likelihood = 0

    # # Loop over the batch
    # for i in range(batch_size):
    #     sentence_log_prob = 0
    #     # Loop over the sequence (ignoring padding)
    #     for j in range(min(data.size(1) - 1, max_len - 1)):  # Use min to avoid out-of-bound indexing
    #         if mask[i, j + 1] == 1:  # If it's not a padding token
    #             word_idx = data[i, j + 1]  # Target token index (shifted by 1)
    #             print('word_idx', word_idx)
    #             # Ensure that word_idx is within vocabulary size
    #             if word_idx >= vocab_size:
    #                 print(f"Warning: word_idx {word_idx} out of vocab size {vocab_size}")
    #                 continue
                
    #             # Add the log-probability of the word to the sentence log probability
    #             sentence_log_prob += log_probs[i, j, word_idx]
    #             print('log',log_probs[i, j, word_idx])
    #     # Add the sentence log probability to the total log-likelihood
    #     log_likelihood += sentence_log_prob
    
    # return log_likelihood
    

In [399]:


model.eval()
correct_predictions = 0
total_predictions = 0
#Forward pass with hidden state update word by word
with torch.no_grad():
    for batch in test_dataloader:

        sentence_good = batch['sentence_good']
        sentence_bad = batch['sentence_bad']

        good = batch['encoded_good']
        bad = batch['encoded_bad']
    
        batch_size = good.size(0)
        seq_len_good = [len(seq)for seq in good ]
        seq_len_bad = [len(seq) for seq in bad]
        max_len_both = [max(seq_len_good), max(seq_len_bad)]
        max_len = max(max_len_both)
        
        hidden_good = model.init_hidden(batch_size)
        hidden_bad = model.init_hidden(batch_size)        
        seq_nll_good = compute_seq_nll(good, hidden_good, batch_size, max_len)
        seq_nll_bad = compute_seq_nll(bad, hidden_bad, batch_size, max_len)
        predictions = (seq_nll_good > seq_nll_bad).cpu().numpy()
        correct_predictions += np.sum(predictions)
        total_predictions += batch_size
        break
        
accuracy = correct_predictions / total_predictions
print(f"Accuracy on {test_dataloader.dataset.dataset.config_name}: {accuracy * 100:.2f}%")
        
        
        

data tensor([[  146,    62, 14479,  ...,    53, 49789,     0],
        [22132, 11047,  5010,  ...,     0,     0,     0],
        [  146, 27491,  3283,  ...,     0,     0,     0],
        ...,
        [ 9906,  5332,   212,  ...,    53, 27571,     0],
        [11686, 11047,     0,  ...,     0,     0,     0],
        [18227, 14479,     3,  ...,     0,     0,     0]])
pad torch.Size([8, 512])
output torch.Size([8, 512, 50001])
targets torch.Size([512, 7])
nll1 torch.Size([3584])
nll2 torch.Size([512, 7])
data tensor([[  146,    62,  8441,  ...,    53, 49789,     0],
        [22132,  3533,  5010,  ...,     0,     0,     0],
        [  146, 27491,  9289,  ...,     0,     0,     0],
        ...,
        [ 9906,  5332,  1513,  ...,    53, 27571,     0],
        [11686,  3533,     0,  ...,     0,     0,     0],
        [18227,  8441,     3,  ...,     0,     0,     0]])
pad torch.Size([8, 512])
output torch.Size([8, 512, 50001])
targets torch.Size([512, 7])
nll1 torch.Size([3584])
nll2 torch.Siz